In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from os import path
from scipy import stats
from tqdm.notebook import tqdm

import matplotlib

from pymodulon.io import load_json_model
from pymodulon.plotting import *

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

# Import Data

In [ ]:
# NOTE
# Currently testing with logTPM for initial pass
# May want to import raw counts so that we can use the same DESeq2 normalization as we did for the original samples

In [ ]:
# import the ICA Data Structure
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
# import TPM data for all samples for downstream activity calculations and metadata
logTPM_file = path.join('../../data/interim', "log_tpm.tsv") # Enter log-TPM filename here
counts_file = path.join('../../data/raw_data/salmon.merged.gene_counts_length_scaled.tsv')
all_metadata_file = path.join('../../data/','interim/','metadata_qc_part1_all.tsv') # Enter full metadata filename here
metadata_file = path.join("../../data/interim/metadata_qc_part1_curated.tsv") # Enter curated metadata filename here

In [ ]:
DF_log_tpm = pd.read_csv(logTPM_file,index_col=[0], sep='\t').fillna(0)
DF_counts = pd.read_csv(counts_file, index_col=0, sep = '\t').fillna(0)
DF_metadata = pd.read_csv(metadata_file,index_col=0, sep='\t')[:-1]
DF_metadata_all = pd.read_csv(all_metadata_file,index_col=0,sep='\t')

In [ ]:
DF_log_tpm = DF_log_tpm[[x for x in DF_log_tpm.columns if x in DF_metadata.index]]
DF_counts = DF_counts[[x for x in DF_counts.columns if x in DF_metadata.index]]

In [ ]:
DF_log_tpm.index = [x[5:] for x in DF_log_tpm.index]
DF_log_tpm = DF_log_tpm.loc[ica_data.M.index]

DF_counts.index = [x[5:] for x in DF_counts.index]
DF_counts = DF_counts.loc[ica_data.M.index]

In [ ]:
# # Test removal of the two large outlier projects
# bioprojects_for_removal = ['PRJNA981303', 'PRJNA541833']

# DF_metadata = DF_metadata[~DF_metadata.BioProject.isin(bioprojects_for_removal)]

# DF_counts = DF_counts.loc[:,DF_metadata.index]

# Centering of data for downstream activity calculations

In [ ]:
# -----------------------------
# ORIGINAL TRAINING DATA
# genes x samples
# -----------------------------
counts_train = DF_counts[ica_data.A.columns]


# Compute geometric means from ORIGINAL data only
geom_means = np.exp(
    np.log(counts_train.replace(0, np.nan)).mean(axis=1)
)

# -----------------------------
# FULL DATASET
# genes x expanded samples
# -----------------------------
counts_full = DF_counts

# Compute ratios
ratios = counts_full.div(geom_means, axis=0)

# Ignore inf/nan from zeros
ratios = ratios.replace([np.inf, -np.inf], np.nan)

# Median-of-ratios size factors
size_factors_full = ratios.median(axis=0)

# Normalize
normalized_full = counts_full.div(size_factors_full, axis=1)

# Log transform
log_normalized_full = np.log2(normalized_full + 1)

In [ ]:
temp_dfs = []
for project, group in DF_metadata_all.loc[log_normalized_full.columns].groupby("BioProject", as_index=False):
    temp_df = log_normalized_full.loc[:,group.index]
    temp_mean = temp_df.mean(axis=1)
    temp_dfs.append(temp_df.sub(temp_mean, axis=0))
    # temp_dfs.append(temp_df.sub(temp_df.iloc[:,0], axis=0))


    
DF_log_tpm_norm = pd.concat(temp_dfs, axis = 1)
DF_log_tpm_norm = DF_log_tpm_norm.loc[log_normalized_full.index]

In [ ]:
from sklearn.cluster import KMeans

original_data_mean = log_normalized_full[ica_data.A.columns].mean(axis=1)

temp_dfs = []
for project, group in DF_metadata_all.loc[log_normalized_full.columns].groupby("BioProject", as_index=False):
    temp_df = log_normalized_full.loc[:, group.index]
    
    # If the project is large enough to have mixed conditions
    if len(group.index) >= 5:
        # Step 1: Use a quick KMeans on the samples (transposed matrix)
        # We look for 2 clusters: e.g., "Baseline-like" vs "Perturbed-like"
        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(temp_df.T)
        labels = kmeans.labels_
        
        # Step 2: Determine which cluster is closer to the original data's mean profile
        mean_profile_0 = temp_df.loc[:, labels == 0].mean(axis=1)
        mean_profile_1 = temp_df.loc[:, labels == 1].mean(axis=1)
        
        # Calculate distance to original data mean vector
        dist_0 = np.linalg.norm(mean_profile_0 - original_data_mean)
        dist_1 = np.linalg.norm(mean_profile_1 - original_data_mean)
        
        # Use the mean of the cluster that represents the "closest to normal" state
        project_inferred_baseline = mean_profile_0 if dist_0 < dist_1 else mean_profile_1
    else:
        project_inferred_baseline = temp_df.mean(axis=1)
        
    temp_dfs.append(temp_df.sub(project_inferred_baseline, axis=0))

DF_log_tpm_norm = pd.concat(temp_dfs, axis=1).loc[log_normalized_full.index]

In [ ]:
# #test with other normalization method
# DF_log_tpm_norm = pd.read_csv('../../expanded_dataset/data/processed_data/log_normalizedCounts_norm.csv', index_col=0)
# DF_log_tpm_norm

In [ ]:
## normalize data using expression mean of all original data instead of project-specific normalization
# original_data_mean = log_normalized_full[ica_data.A.columns].mean(axis=1)
# DF_log_tpm_norm = log_normalized_full.sub(original_data_mean, axis=0)

# Infer Activities

In [ ]:
from pymodulon.util import infer_activities
# Run inference with properly aligned data
activities = infer_activities(ica_data, DF_log_tpm_norm)

# Testing Performance on Existing Samples

In [ ]:
test_X = DF_log_tpm_norm[ica_data.A.columns]
test_activities = activities[ica_data.A.columns]

In [ ]:
baseline = test_X

# Base error (variance of original centered data)
base_err = np.linalg.norm(baseline) ** 2

# Reconstruction: M @ A
MA = ica_data.M.values @ test_activities.values

# Reconstruction error
sa_err = np.linalg.norm(MA - baseline) ** 2

# Variance explained
var_explained = np.clip(1 - sa_err / base_err, 0, 1)

In [ ]:
var_explained

In [ ]:
test_X = DF_log_tpm_norm[[x for x in DF_log_tpm_norm.columns if x not in ica_data.A.columns]]
test_activities = activities[[x for x in DF_log_tpm_norm.columns if x not in ica_data.A.columns]]

baseline = test_X

# Base error (variance of original centered data)
base_err = np.linalg.norm(baseline) ** 2
 
# Reconstruction: M @ A
MA = ica_data.M.values @ test_activities.values

# Reconstruction error
sa_err = np.linalg.norm(MA - baseline) ** 2

# Variance explained
var_explained = np.clip(1 - sa_err / base_err, 0, 1)
var_explained

# Plotting and Analysis of the Inferred Activities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Standardize and run PCA on the inferred activities
scaler = StandardScaler()
activities_scaled = scaler.fit_transform(activities.T)

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(activities_scaled)

# 2. Create the Scores DataFrame
df_scores = pd.DataFrame(
    pca_scores, columns=["PC1", "PC2"], index=activities.columns
)

# 3. Create a column to identify the source of the sample
# We check if the sample name exists in the original ica_data.A columns
original_samples = set(ica_data.A.columns)
df_scores["Sample_Source"] = [
    "Original ICA Dataset" if sample in original_samples else "New Dataset"
    for sample in df_scores.index
]

# 4. Create the Loadings DataFrame (ICs / i-modulons)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
df_loadings = pd.DataFrame(
    loadings, columns=["PC1", "PC2"], index=activities.index
)

# 5. Plotting
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot: Color points by 'Sample_Source'
# We use a distinct palette to easily tell them apart
sns.scatterplot(
    data=df_scores,
    x="PC1",
    y="PC2",
    hue="Sample_Source",
    palette={"Original ICA Dataset": "#4C72B0", "New Dataset": "#DD8452"},
    alpha=0.8,
    s=70,
    edgecolor="w",
    linewidth=0.5,
    ax=ax,
)

# 6. Overlay the arrows for the top N i-modulons
arrow_expansion = 3.0
top_n_arrows = 10
important_ics = (df_loadings["PC1"] ** 2 + df_loadings["PC2"] ** 2).nlargest(
    top_n_arrows
).index

for imodulon in important_ics:
    x = df_loadings.loc[imodulon, "PC1"] * arrow_expansion
    y = df_loadings.loc[imodulon, "PC2"] * arrow_expansion

    # Draw arrow
    ax.arrow(
        0,
        0,
        x,
        y,
        color="crimson",
        alpha=0.6,
        head_width=0.04 * arrow_expansion,
        linewidth=1.2,
    )
    # Label arrow
    ax.text(
        x * 1.1,
        y * 1.1,
        imodulon,
        color="maroon",
        ha="center",
        va="center",
        fontsize=9,
        weight="bold",
    )

# Formatting
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
ax.set_xlabel(f"PC1 ({pc1_var:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC2 ({pc2_var:.1f}%)", fontsize=12)
ax.set_title(
    "PCA Biplot: Original vs. New Inferred Activities", fontsize=14, pad=15
)
ax.grid(True, linestyle="--", alpha=0.4)

# Clean up legend placement
ax.legend(title="Sample Group", loc="best")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PCA on inferred activities
# ============================================================

scaler = StandardScaler()

activities_scaled = scaler.fit_transform(activities.T)

pca = PCA(n_components=2)

scores = pca.fit_transform(activities_scaled)

loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

# ============================================================
# Scores dataframe
# ============================================================

df_scores = pd.DataFrame(
    scores,
    columns=["PC1", "PC2"],
    index=activities.columns
)

# ============================================================
# Assign groups
# ============================================================

original_samples = set(ica_data.A.columns)

# Everything starts as "New inferred"
df_scores["Group"] = "New inferred"

# Original ICA samples
df_scores.loc[
    df_scores.index.isin(original_samples),
    "Group"
] = "Original ICA"

# Highlight one specific project
project_samples = DF_metadata.index[
    DF_metadata["BioProject"] == "PRJNA589326"
]

df_scores.loc[
    df_scores.index.isin(project_samples),
    "Group"
] = "PRJNA589326"

# ============================================================
# Loadings
# ============================================================

df_loadings = pd.DataFrame(
    loadings,
    columns=["PC1", "PC2"],
    index=activities.index
)

top_pc1 = (
    df_loadings["PC1"]
    .abs()
    .nlargest(4)
    .index
)

top_pc2 = (
    df_loadings["PC2"]
    .abs()
    .nlargest(4)
    .index
)

top_imods = list(dict.fromkeys(top_pc1.tolist() + top_pc2.tolist()))

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(10,8))

palette = {
    "Original ICA": "#3b4ac4",
    "New inferred": "#4aae3b",
    "PRJNA589326": "#c43b4a"
}

sizes = {
    "Original ICA": 35,
    "New inferred": 35,
    "PRJNA589326": 50
}

alphas = {
    "Original ICA": 0.6,
    "New inferred": 0.3,
    "PRJNA589326": 1.00
}

# Plot each group separately for better control

for group in [
    "Original ICA",
    "New inferred",
    "PRJNA589326"
]:

    sub = df_scores[df_scores["Group"] == group]

    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        color=palette[group],
        s=sizes[group],
        alpha=alphas[group],
        label=group,
        edgecolor="black" if group=="PRJNA589326" else "none",
        linewidth=0.4,
        rasterized=True
    )



# ============================================================
# iModulon arrows
# ============================================================

arrow_scale = 12

for imod in top_imods:

    x = df_loadings.loc[imod,"PC1"] * arrow_scale
    y = df_loadings.loc[imod,"PC2"] * arrow_scale

    ax.annotate(
        "",
        xy=(x,y),
        xytext=(0,0),
        arrowprops=dict(
            color="darkred",
            lw=2,
            arrowstyle="->"
        )
    )

    dx = 0.2*np.sign(x)
    dy = 0.2*np.sign(y)

    ax.text(
        x+dx,
        y+dy,
        imod,
        fontsize=10,
        weight="bold",
        color="darkred"
    )

# ============================================================
# Formatting
# ============================================================

pc1 = 100*pca.explained_variance_ratio_[0]
pc2 = 100*pca.explained_variance_ratio_[1]

ax.set_xlabel(f"PC1 ({pc1:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pc2:.1f}% variance)")

ax.set_title(
    "PCA of inferred iModulon activities for Bulk Samples"
)

ax.grid(
    linestyle="--",
    alpha=0.3
)

ax.legend(
    frameon=True,
    title=""
)

plt.tight_layout()
plt.savefig('images/fig6_inferred_activity_pca.svg', dpi = 600, format = 'svg', bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# 1. Reconstruct the matrix
X_actual = DF_log_tpm_norm.loc[ica_data.M.index, [x for x in activities.columns if x not in ica_data.A.columns]]
X_recon = pd.DataFrame(
    np.dot(ica_data.M.values, activities[[x for x in activities.columns if x not in ica_data.A.columns]].values),
    index=X_actual.index,
    columns=X_actual.columns,
)

# 2. Calculate both R2 and Pearson Correlation for every sample
performance_metrics = []
for sample in [x for x in activities.columns if x not in ica_data.A.columns]:
    # Calculate R2 using the original global-mean centered data you ran
    r2 = r2_score(X_actual[sample], X_recon[sample])
    # Calculate Pearson correlation
    corr, _ = pearsonr(X_actual[sample], X_recon[sample])

    performance_metrics.append(
        {"Sample": sample, "R2": r2, "Pearson_r": corr}
    )

df_perf = pd.DataFrame(performance_metrics)

# 3. Plotting the Global Audit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of R2 values
sns.histplot(df_perf["R2"], bins=30, kde=True, ax=axes[0], color="#DEe2e6")
axes[0].axvline(
    0, color="red", linestyle="--", label="Threshold for 'Worse than Mean'"
)
axes[0].set_title("Distribution of $R^2$ across Samples")
axes[0].set_xlabel("$R^2$ Score")
axes[0].legend()

# Plot 2: Distribution of Pearson Correlation
sns.histplot(df_perf["Pearson_r"], bins=30, kde=True, ax=axes[1], color="#4C72B0")
axes[1].axvline(
    0.5, color="orange", linestyle="--", label="Moderate Correlation (0.5)"
)
axes[1].set_title("Distribution of Pearson Correlation ($r$)")
axes[1].set_xlabel("Pearson Correlation Coefficient")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print Global Summary statistics
print(f"Total Samples evaluated: {len(df_perf)}")
print(
    f"Samples with positive R2: {sum(df_perf['R2'] > 0)} ({sum(df_perf['R2'] > 0)/len(df_perf)*100:.1f}%)"
)
print(
    f"Samples with Pearson r > 0.5: {sum(df_perf['Pearson_r'] > 0.5)} ({sum(df_perf['Pearson_r'] > 0.5)/len(df_perf)*100:.1f}%)"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# 1. Reconstruct the matrix
X_actual = DF_log_tpm_norm.loc[ica_data.M.index, [x for x in activities.columns if x in ica_data.A.columns]]
X_recon = pd.DataFrame(
    np.dot(ica_data.M.values, activities[[x for x in activities.columns if x in ica_data.A.columns]].values),
    index=X_actual.index,
    columns=X_actual.columns,
)

# 2. Calculate both R2 and Pearson Correlation for every sample
performance_metrics = []
for sample in [x for x in activities.columns if x in ica_data.A.columns]:
    # Calculate R2 using the original global-mean centered data you ran
    r2 = r2_score(X_actual[sample], X_recon[sample])
    # Calculate Pearson correlation
    corr, _ = pearsonr(X_actual[sample], X_recon[sample])

    performance_metrics.append(
        {"Sample": sample, "R2": r2, "Pearson_r": corr}
    )

df_perf = pd.DataFrame(performance_metrics)

# 3. Plotting the Global Audit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of R2 values
sns.histplot(df_perf["R2"], bins=30, kde=True, ax=axes[0], color="#DEe2e6")
axes[0].axvline(
    0, color="red", linestyle="--", label="Threshold for 'Worse than Mean'"
)
axes[0].set_title("Distribution of $R^2$ across Samples")
axes[0].set_xlabel("$R^2$ Score")
axes[0].legend()

# Plot 2: Distribution of Pearson Correlation
sns.histplot(df_perf["Pearson_r"], bins=30, kde=True, ax=axes[1], color="#4C72B0")
axes[1].axvline(
    0.5, color="orange", linestyle="--", label="Moderate Correlation (0.5)"
)
axes[1].set_title("Distribution of Pearson Correlation ($r$)")
axes[1].set_xlabel("Pearson Correlation Coefficient")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print Global Summary statistics
print(f"Total Samples evaluated: {len(df_perf)}")
print(
    f"Samples with positive R2: {sum(df_perf['R2'] > 0)} ({sum(df_perf['R2'] > 0)/len(df_perf)*100:.1f}%)"
)
print(
    f"Samples with Pearson r > 0.5: {sum(df_perf['Pearson_r'] > 0.5)} ({sum(df_perf['Pearson_r'] > 0.5)/len(df_perf)*100:.1f}%)"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

# 1. Reconstruct the matrix
X_actual = DF_log_tpm_norm.loc[ica_data.M.index, activities.columns]
X_recon = pd.DataFrame(
    np.dot(ica_data.M.values, activities.values),
    index=X_actual.index,
    columns=X_actual.columns,
)

# 2. Calculate both R2 and Pearson Correlation for every sample
performance_metrics = []
for sample in activities.columns:
    # Calculate R2 using the original global-mean centered data you ran
    r2 = r2_score(X_actual[sample], X_recon[sample])
    # Calculate Pearson correlation
    corr, _ = pearsonr(X_actual[sample], X_recon[sample])

    performance_metrics.append(
        {"Sample": sample, "R2": r2, "Pearson_r": corr}
    )

df_perf = pd.DataFrame(performance_metrics)

# 3. Plotting the Global Audit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of R2 values
sns.histplot(df_perf["R2"], bins=30, kde=True, ax=axes[0], color="#DEe2e6")
axes[0].axvline(
    0, color="red", linestyle="--", label="Threshold for 'Worse than Mean'"
)
axes[0].set_title("Distribution of $R^2$ across Samples")
axes[0].set_xlabel("$R^2$ Score")
axes[0].legend()

# Plot 2: Distribution of Pearson Correlation
sns.histplot(df_perf["Pearson_r"], bins=30, kde=True, ax=axes[1], color="#4C72B0")
axes[1].axvline(
    0.5, color="orange", linestyle="--", label="Moderate Correlation (0.5)"
)
axes[1].set_title("Distribution of Pearson Correlation ($r$)")
axes[1].set_xlabel("Pearson Correlation Coefficient")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print Global Summary statistics
print(f"Total Samples evaluated: {len(df_perf)}")
print(
    f"Samples with positive R2: {sum(df_perf['R2'] > 0)} ({sum(df_perf['R2'] > 0)/len(df_perf)*100:.1f}%)"
)
print(
    f"Samples with Pearson r > 0.5: {sum(df_perf['Pearson_r'] > 0.5)} ({sum(df_perf['Pearson_r'] > 0.5)/len(df_perf)*100:.1f}%)"
)

In [ ]:
df_perf["BioProject"] = (
    DF_metadata_all.loc[df_perf["Sample"], "BioProject"].values
)

df_perf["Original_data"] = (
    df_perf.BioProject.apply(lambda x: x in ica_data.sample_table.BioProject.unique())
)

df_perf["Condition"] = (
    DF_metadata.loc[df_perf["Sample"], "condition"].values
)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

highlight_projects = {'PRJNA981303', 'PRJNA541833'}
ica_samples = set(ica_data.A.columns)

df_plot = df_perf.copy()

df_plot['color_group'] = 'Other New Data'

df_plot.loc[
    df_plot['Sample'].isin(ica_samples),
    'color_group'
] = 'Original Data'

df_plot.loc[
    df_plot['BioProject'].isin(highlight_projects),
    'color_group'
] = 'WIS Projects'

color_map = {
    'Original Data': 'green',
    'WIS Projects': 'red',
    'Other New Data': 'gray'
}

# Create subplot:
# left = strip plot
# right = histograms
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    column_widths=[0.8, 0.2],
    horizontal_spacing=0.03
)

# -----------------------
# Strip plot
# -----------------------
for group, color in color_map.items():

    d = df_plot[df_plot['color_group'] == group]

    fig.add_trace(
        go.Box(
            x=d['BioProject'],
            y=d['R2'],
            name=group,
            marker_color=color,
            boxpoints='all',
            jitter=0.4,
            pointpos=0,
            fillcolor='rgba(0,0,0,0)',
            line=dict(width=0),
            showlegend=True,

            customdata=d[['Sample', 'Condition', 'BioProject']],

            hovertemplate=(
                "Sample: %{customdata[0]}<br>"
                "Condition: %{customdata[1]}<br>"
                "BioProject: %{customdata[2]}<br>"
                "R²: %{y:.3f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1
    )
# -----------------------
# Histograms
# -----------------------
for group, color in color_map.items():

    d = df_plot[df_plot['color_group'] == group]

    fig.add_trace(
        go.Histogram(
            y=d['R2'],
            name=group,
            marker_color=color,
            opacity=0.6,
            histnorm='probability density',
            nbinsy=30,
            orientation='h',
            showlegend=False
        ),
        row=1,
        col=2
    )

fig.update_layout(
    width=1200,
    height=600,
    barmode='overlay',
)

fig.update_xaxes(
    tickangle=-90,
    row=1,
    col=1
)

fig.update_xaxes(
    title='Density',
    row=1,
    col=2
)

fig.update_yaxes(
    title='R2'
)

fig.show()

# Analysis of Residuals from Reconstruction

In [ ]:
R = X_actual - X_recon

In [ ]:
residual_norm = np.linalg.norm(R, axis=0)

df_perf["ResidualNorm"] = residual_norm

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)

R_pca = pca.fit_transform(R.T)

explained = pca.explained_variance_ratio_

In [ ]:
pc1_loadings = pd.Series(
    pca.components_[0],
    index=R.index
).sort_values()

In [ ]:
df_perf["BioProject"] = (
    DF_metadata_all.loc[df_perf["Sample"], "BioProject"].values
)

df_perf["Original_data"] = (
    df_perf.BioProject.apply(lambda x: x in ica_data.sample_table.BioProject.unique())
)

In [ ]:
pd.concat(
    [
        df_perf.groupby('BioProject')['R2'].mean().rename('median_R2'),
        df_perf['BioProject'].value_counts().rename('n'),
        df_perf.groupby('BioProject')['Original_data']
               .first()
               .rename('Original_data')
    ],
    axis=1
).sort_values('median_R2')

# Analyze a New Project of Interest

In [ ]:
target_project = 'PRJNA589326'
project_metadata = DF_metadata[DF_metadata.BioProject == target_project].copy()
project_activities = activities[project_metadata.index].copy()

# project_metadata.loc[:,'condition'] = project_metadata.LibraryName.fillna('').apply(lambda x: x.rsplit('_', maxsplit=1)[0])

In [ ]:
project_conditions = {
'SRX7135755' : 'wt_t0' ,
'SRX7135756' : 'wt_t2' ,
'SRX7135757' : 'wt_t4' ,
'SRX7135758' : 'wt_t5' ,
'SRX7135759' : 'wt_t6' ,
'SRX7135760' : 'wt_t7' ,
'SRX7135761' : 'wt_t8' ,
'SRX7135762' : 'wt_t9' ,
'SRX7135763' : 'wt_t10' ,
'SRX7135764' : 'wt_t11' ,
'SRX7135765' : 'wt_t12' ,
'SRX7135766' : 'wt_t13' ,
'SRX7135767' : 'wt_t14' ,
'SRX7135768' : 'wt_t15' ,
'SRX7135769' : 'wt_t16' ,
'SRX7135770' : 'wt_t18' ,
'SRX7135771' : 'wt_t20' ,
'SRX7135772' : 'wt_t22' ,
'SRX7135773' : 'wt_t24' ,
'SRX7135774' : 'wt_t28' ,
'SRX7135775' : 'wt_t32' ,
'SRX7135776' : 'wt_t38' ,
'SRX7135777' : 'wt_t46' ,
'SRX7135778' : 'wt_t58' ,
'SRX7135779' : 'nde1d_nde2d_ndi1d_t0' ,
'SRX7135780' : 'nde1d_nde2d_ndi1d_t2' ,
'SRX7135781' : 'nde1d_nde2d_ndi1d_t4' ,
'SRX7135782' : 'nde1d_nde2d_ndi1d_t5' ,
'SRX7135783' : 'nde1d_nde2d_ndi1d_t6' ,
'SRX7135784' : 'nde1d_nde2d_ndi1d_t7' ,
'SRX7135785' : 'nde1d_nde2d_ndi1d_t8' ,
'SRX7135786' : 'nde1d_nde2d_ndi1d_t9' ,
'SRX7135787' : 'nde1d_nde2d_ndi1d_t10' ,
'SRX7135788' : 'nde1d_nde2d_ndi1d_t11' ,
'SRX7135789' : 'nde1d_nde2d_ndi1d_t12' ,
'SRX7135790' : 'nde1d_nde2d_ndi1d_t13' ,
'SRX7135791' : 'nde1d_nde2d_ndi1d_t14' ,
'SRX7135792' : 'nde1d_nde2d_ndi1d_t15' ,
'SRX7135793' : 'nde1d_nde2d_ndi1d_t16' ,
'SRX7135794' : 'nde1d_nde2d_ndi1d_t18' ,
'SRX7135795' : 'nde1d_nde2d_ndi1d_t20' ,
'SRX7135796' : 'nde1d_nde2d_ndi1d_t22' ,
'SRX7135797' : 'nde1d_nde2d_ndi1d_t24' ,
'SRX7135798' : 'nde1d_nde2d_ndi1d_t28' ,
'SRX7135799' : 'nde1d_nde2d_ndi1d_t32' ,
'SRX7135800' : 'nde1d_nde2d_ndi1d_t38' ,
'SRX7135801' : 'nde1d_nde2d_ndi1d_t46' ,
'SRX7135802' : 'nde1d_nde2d_ndi1d_t58' ,
'SRX7135803' : 'cox4d_t0' ,
'SRX7135804' : 'cox4d_t2' ,
'SRX7135805' : 'cox4d_t4' ,
'SRX7135806' : 'cox4d_t5' ,
'SRX7135807' : 'cox4d_t6' ,
'SRX7135808' : 'cox4d_t7' ,
'SRX7135809' : 'cox4d_t8' ,
'SRX7135810' : 'cox4d_t9' ,
'SRX7135811' : 'cox4d_t10' ,
'SRX7135812' : 'cox4d_t11' ,
'SRX7135813' : 'cox4d_t12' ,
'SRX7135814' : 'cox4d_t13' ,
'SRX7135815' : 'cox4d_t14' ,
'SRX7135816' : 'cox4d_t15' ,
'SRX7135817' : 'cox4d_t16' ,
'SRX7135818' : 'cox4d_t18' ,
'SRX7135819' : 'cox4d_t20' ,
'SRX7135820' : 'cox4d_t22' ,
'SRX7135821' : 'cox4d_t24' ,
'SRX7135822' : 'cox4d_t28' ,
'SRX7135823' : 'cox4d_t32' ,
'SRX7135824' : 'cox4d_t38' ,
'SRX7135825' : 'cox4d_t46' ,
'SRX7135826' : 'cox4d_t58' ,
}

In [ ]:
project_metadata['condition'] = pd.Series(project_conditions)

In [ ]:
project_metadata['full_name'] = project_metadata.project + ':' + project_metadata.condition

In [ ]:
# Samples x iModulons
X = project_activities.T

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
scores = pca.fit_transform(X_scaled)

# Sample scores
df_scores = pd.DataFrame(
    scores,
    columns=["PC1", "PC2"],
    index=X.index,
)

# Add metadata
# Add metadata
df_scores = df_scores.join(project_metadata[["condition"]])

# Split condition into deletion and timepoint
df_scores["deletion"] = df_scores["condition"].str.rsplit("_", n=1).str[0]
df_scores["timepoint"] = df_scores["condition"].str.rsplit("_", n=1).str[1]


# Loadings
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
df_loadings = pd.DataFrame(
    loadings,
    columns=["PC1", "PC2"],
    index=project_activities.index,
)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

# Order timepoints numerically
time_order = sorted(
    df_scores["timepoint"].unique(),
    key=lambda x: int(x[1:])  # remove the leading "t"
)

sns.scatterplot(
    data=df_scores,
    x="PC1",
    y="PC2",
    hue="timepoint",
    hue_order=time_order,
    style="deletion",
    s=100,
    palette="magma",
    ax=ax,
)

# Top contributing iModulons
arrow_expansion = 3
top_n_arrows = 5

important_ics = (
    df_loadings["PC1"]**2 + df_loadings["PC2"]**2
).nlargest(top_n_arrows).index

for imodulon in important_ics:
    x = df_loadings.loc[imodulon, "PC1"] * arrow_expansion
    y = df_loadings.loc[imodulon, "PC2"] * arrow_expansion

    ax.arrow(
        0, 0, x, y,
        color="crimson",
        alpha=0.5,
        head_width=0.05,
        linewidth=1.2,
    )

    ax.text(
        x * 1.1,
        y * 1.1,
        imodulon,
        fontsize=9,
        color="maroon",
        ha="center",
    )

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100


ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0,
    frameon=True,
)


ax.set_xlabel(f"PC1 ({pc1_var:.1f}%)")
ax.set_ylabel(f"PC2 ({pc2_var:.1f}%)")
ax.set_title(f"PCA of iModulon Activities: {target_project}")

plt.tight_layout()
plt.show()

In [ ]:
# Samples x iModulons
X = project_activities.T.copy()

# Add metadata
X = X.join(project_metadata[["condition"]])

# Split metadata
X["deletion"] = X["condition"].str.rsplit("_", n=1).str[0]
X["timepoint"] = X["condition"].str.rsplit("_", n=1).str[1]

# Numeric time
X["time_numeric"] = (
    X["timepoint"]
    .astype(str)
    .str.extract(r"t(\d+)")
    .astype(int)
)

# Order timepoints numerically
time_order = sorted(
    X["timepoint"].unique(),
    key=lambda x: int(x[1:])   # assumes t0, t15, t60, ...
)

# Sort samples by deletion then timepoint
X["timepoint"] = pd.Categorical(
    X["timepoint"],
    categories=time_order,
    ordered=True,
)

X = X.sort_values(["deletion", "timepoint"])

# Extract activity matrix
activity = X.drop(columns=["condition", "deletion", "timepoint"])

# Optional: z-score each iModulon across samples
activity_scaled = pd.DataFrame(
    StandardScaler().fit_transform(activity),
    index=activity.index,
    columns=activity.columns,
)

# Row colors
deletion_palette = dict(
    zip(
        X["deletion"].unique(),
        sns.color_palette("tab10", X["deletion"].nunique())
    )
)

time_palette = dict(
    zip(
        time_order,
        sns.color_palette("magma", len(time_order))
    )
)

row_colors = pd.DataFrame({
    "Deletion": X["deletion"].map(deletion_palette),
    "Time": X["timepoint"].astype(str).map(time_palette),
}, index=activity.index)

# Clustermap
g = sns.clustermap(
    activity_scaled,
    cmap="vlag",
    center=0,
    figsize=(16, 10),
    row_cluster=False,
    col_cluster=True,
    row_colors=row_colors,
    dendrogram_ratio=(0.15, 0.08),
    colors_ratio=0.03,
    cbar_pos=(1.1, 0.35, 0.06, 0.55),
    xticklabels=True,
    yticklabels=False,   # often cleaner with many samples
)

from matplotlib.patches import Patch


# Deletion legend
deletion_handles = [
    Patch(facecolor=color, label=deletion)
    for deletion, color in deletion_palette.items()
]

# Timepoint legend
time_handles = [
    Patch(facecolor=color, label=time)
    for time, color in time_palette.items()
]

legend1 = g.ax_heatmap.legend(
    handles=deletion_handles,
    title="Deletion",
    bbox_to_anchor=(-0.15, 1.2),
    loc="upper right",
    frameon=True,
)

g.ax_heatmap.add_artist(legend1)

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Numeric times
times = X["time_numeric"]

norm = Normalize(vmin=times.min(), vmax=times.max())
sm = ScalarMappable(cmap="magma", norm=norm)
sm.set_array([])

cax = g.fig.add_axes([.04, 0.35, 0.03, 0.6])  # left, bottom, width, height
cb = g.fig.colorbar(sm, cax=cax)
cb.set_label("Time")

unique_times = np.sort(X["time_numeric"].unique())
cb.ax.tick_params(labelsize=9)
cb.set_ticks(unique_times)
cb.set_ticklabels(unique_times)


g.ax_heatmap.set_xlabel("iModulons")
g.ax_heatmap.set_ylabel("Samples")

plt.savefig('images/fig6_bioproject_analysis_full.svg', dpi = 600, format = 'svg', bbox_inches='tight')

In [ ]:
from scipy.stats import spearmanr
import pandas as pd

results = []

for imod in activity.columns:
    rho, p = spearmanr(activity[imod], X["time_numeric"])
    results.append({
        "iModulon": imod,
        "rho": rho,
        "pvalue": p,
        "abs_rho": abs(rho),
    })

corr_df = (
    pd.DataFrame(results)
      .sort_values("abs_rho", ascending=False)
)

In [ ]:
# Keep only strongly correlated iModulons
selected_imods = corr_df.loc[
    corr_df["abs_rho"] >= 0.7,
    "iModulon"
]

activity_subset = activity_scaled[selected_imods]

g = sns.clustermap(
    activity_subset,
    cmap="vlag",
    center=0,
    figsize=(10, 10),
    row_cluster=False,
    col_cluster=True,
    row_colors=row_colors,
    dendrogram_ratio=(0.15, 0.08),
    colors_ratio=0.03,
    cbar_pos=(1.07, 0.35, 0.04, 0.55),
    xticklabels=True,
    yticklabels=False,
)


# Deletion legend
deletion_handles = [
    Patch(facecolor=color, label=deletion)
    for deletion, color in deletion_palette.items()
]

# Timepoint legend
time_handles = [
    Patch(facecolor=color, label=time)
    for time, color in time_palette.items()
]

legend1 = g.ax_heatmap.legend(
    handles=deletion_handles,
    title="Deletion",
    bbox_to_anchor=(-0.15, 1.2),
    loc="upper right",
    frameon=True,
)

g.ax_heatmap.add_artist(legend1)

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Numeric times
times = X["time_numeric"]

norm = Normalize(vmin=times.min(), vmax=times.max())
sm = ScalarMappable(cmap="magma", norm=norm)
sm.set_array([])

cax = g.fig.add_axes([.04, 0.35, 0.03, 0.6])  # left, bottom, width, height
cb = g.fig.colorbar(sm, cax=cax)
cb.set_label("Time")

unique_times = np.sort(X["time_numeric"].unique())
cb.ax.tick_params(labelsize=9)
cb.set_ticks(unique_times)
cb.set_ticklabels(unique_times)


g.ax_heatmap.set_xlabel("iModulons")
g.ax_heatmap.set_ylabel("Samples")

plt.savefig('images/fig6_bioproject_analysis.svg', dpi = 600, format = 'svg', bbox_inches='tight')
plt.show()

# Test Performance of Original ICA Decomposition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

# 1. Reconstruct the matrix
X_actual = ica_data.X.loc[ica_data.M.index, ica_data.A.columns]
X_recon = pd.DataFrame(
    np.dot(ica_data.M.values, ica_data.A.values),
    index=X_actual.index,
    columns=X_actual.columns,
)

# 2. Calculate both R2 and Pearson Correlation for every sample
performance_metrics = []
for sample in ica_data.A.columns:
    # Calculate R2 using the original global-mean centered data you ran
    r2 = r2_score(X_actual[sample], X_recon[sample])
    # Calculate Pearson correlation
    corr, _ = pearsonr(X_actual[sample], X_recon[sample])

    performance_metrics.append(
        {"Sample": sample, "R2": r2, "Pearson_r": corr}
    )

df_perf = pd.DataFrame(performance_metrics)

# 3. Plotting the Global Audit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of R2 values
sns.histplot(df_perf["R2"], bins=30, kde=True, ax=axes[0], color="#DEe2e6")
axes[0].axvline(
    0, color="red", linestyle="--", label="Threshold for 'Worse than Mean'"
)
axes[0].set_title("Distribution of $R^2$ across Samples")
axes[0].set_xlabel("$R^2$ Score")
axes[0].legend()

# Plot 2: Distribution of Pearson Correlation
sns.histplot(df_perf["Pearson_r"], bins=30, kde=True, ax=axes[1], color="#4C72B0")
axes[1].axvline(
    0.5, color="orange", linestyle="--", label="Moderate Correlation (0.5)"
)
axes[1].set_title("Distribution of Pearson Correlation ($r$)")
axes[1].set_xlabel("Pearson Correlation Coefficient")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print Global Summary statistics
print(f"Total Samples evaluated: {len(df_perf)}")
print(
    f"Samples with positive R2: {sum(df_perf['R2'] > 0)} ({sum(df_perf['R2'] > 0)/len(df_perf)*100:.1f}%)"
)
print(
    f"Samples with Pearson r > 0.5: {sum(df_perf['Pearson_r'] > 0.5)} ({sum(df_perf['Pearson_r'] > 0.5)/len(df_perf)*100:.1f}%)"
)

In [ ]:
df_perf["project"] = (
    ica_data.sample_table.loc[df_perf["Sample"], "BioProject"].values
)

df_perf["Original_data"] = (
    df_perf.project.apply(lambda x: x in ica_data.sample_table.BioProject.unique())
)

performance_readouts = pd.concat(
    [
        df_perf.groupby('project')['R2'].mean().rename('median_R2'),
        df_perf['project'].value_counts().rename('n'),
        df_perf.groupby('project')['Original_data']
               .first()
               .rename('Original_data')
    ],
    axis=1
).sort_values('median_R2')

In [ ]:
performance_readouts

In [ ]:
test_X = ica_data.X[ica_data.A.columns]
test_activities = ica_data.A[ica_data.A.columns]

baseline = test_X

# Base error (variance of original centered data)
base_err = np.linalg.norm(baseline) ** 2

# Reconstruction: M @ A
MA = ica_data.M.values @ test_activities.values

# Reconstruction error
sa_err = np.linalg.norm(MA - baseline) ** 2

# Variance explained
var_explained = np.clip(1 - sa_err / base_err, 0, 1)
var_explained

In [ ]:
# ------------------------------------------------------------
# Split activity matrix
# ------------------------------------------------------------

original_samples = list(ica_data.A.columns)

activities_original = activities.loc[:, original_samples]

activities_new = activities.loc[:, [x for x in activities.columns if x not in original_samples]]

# ------------------------------------------------------------
# Standardize using only original ICA samples
# ------------------------------------------------------------

scaler = StandardScaler()

original_scaled = scaler.fit_transform(activities_original.T)

# ------------------------------------------------------------
# Learn PCA basis from original ICA compendium
# ------------------------------------------------------------

pca = PCA(n_components=2)

pca.fit(original_scaled)

# ------------------------------------------------------------
# Project all samples into the learned PCA space
# ------------------------------------------------------------

all_scaled = scaler.transform(activities.T)

scores = pca.transform(all_scaled)

loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

df_scores = pd.DataFrame(
    scores,
    columns=["PC1", "PC2"],
    index=activities.columns
)

palette = {
    "Original ICA": "#BFBFBF",
    "New inferred": "#2A6FBB",
    "PRJNA589326": "#D62728"
}

sizes = {
    "Original ICA": 50,
    "New inferred": 20,
    "PRJNA589326": 55
}

alphas = {
    "Original ICA": 0.4,
    "New inferred": 0.55,
    "PRJNA589326": 1.00
}

# ============================================================
# Assign groups
# ============================================================

df_scores["Group"] = "New inferred"

df_scores.loc[
    df_scores.index.isin(original_samples),
    "Group"
] = "Original ICA"

project_samples = DF_metadata.index[
    DF_metadata["BioProject"] == "PRJNA589326"
]

df_scores.loc[
    df_scores.index.isin(project_samples),
    "Group"
] = "PRJNA589326"


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

plot_order = [
    "Original ICA",
    "New inferred",
    "PRJNA589326"
]

for group in plot_order:

    sub = df_scores[df_scores["Group"] == group]

    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        color=palette[group],
        s=sizes[group],
        alpha=alphas[group],
        label=group,
        edgecolors="black" if group == "PRJNA589326" else "none",
        linewidth=0.6,
        rasterized=True,
        zorder=3 if group == "PRJNA589326" else 2 if group == "New inferred" else 1
    )


# ============================================================
# Reference axes
# ============================================================

ax.axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=0.8,
    alpha=0.4
)

ax.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=0.8,
    alpha=0.4
)



# ============================================================
# Loading dataframe
# ============================================================

df_loadings = pd.DataFrame(
    loadings,
    columns=["PC1", "PC2"],
    index=activities.index
)


# ============================================================
# Select top contributors to each PC
# ============================================================

top_pc1_pos = df_loadings["PC1"].nlargest(3).index
top_pc1_neg = df_loadings["PC1"].nsmallest(3).index

top_pc2_pos = df_loadings["PC2"].nlargest(3).index
top_pc2_neg = df_loadings["PC2"].nsmallest(3).index


top_imods = list(
    top_pc1_pos
) + list(
    top_pc1_neg
) + list(
    top_pc2_pos
) + list(
    top_pc2_neg
)

# Remove duplicates if an iModulon contributes to multiple axes
top_imods = list(dict.fromkeys(top_imods))

# ============================================================
# Add biplot arrows
# ============================================================

arrow_scale = 4.0

for imod in top_imods:

    x = df_loadings.loc[imod, "PC1"] * arrow_scale
    y = df_loadings.loc[imod, "PC2"] * arrow_scale

    ax.annotate(
        "",
        xy=(x, y),
        xytext=(0, 0),
        arrowprops=dict(
            color="darkred",
            linewidth=1.8,
            arrowstyle="->",
            alpha=0.8
        ),
        zorder=5
    )

    # label offset
    offset_x = 0.03 * (ax.get_xlim()[1] - ax.get_xlim()[0])
    offset_y = 0.03 * (ax.get_ylim()[1] - ax.get_ylim()[0])

    ax.text(
        x + offset_x*np.sign(x if x != 0 else 1),
        y + offset_y*np.sign(y if y != 0 else 1),
        imod,
        fontsize=9,
        color="darkred",
        weight="bold",
        zorder=6
    )


# ============================================================
# Explained variance
# ============================================================

pc1 = 100 * pca.explained_variance_ratio_[0]
pc2 = 100 * pca.explained_variance_ratio_[1]

ax.set_xlabel(f"Reference PC1 ({pc1:.1f}% variance)")
ax.set_ylabel(f"Reference PC2 ({pc2:.1f}% variance)")


# ============================================================
# Labels
# ============================================================

ax.set_title(
    "Projection of inferred iModulon activities into the reference ICA activity space"
)


# ============================================================
# Legend
# ============================================================

leg = ax.legend(
    frameon=False,
    loc="best"
)

# ============================================================
# Cosmetic formatting
# ============================================================

ax.grid(
    linestyle=":",
    alpha=0.25
)

sns.despine()

plt.tight_layout()

plt.savefig(
    "images/reference_activity_projection.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Analyze Distribution of Samples Excluded from ICA

In [ ]:
DF_metadata_new = DF_metadata.loc[[x for x in DF_metadata.index if x not in ica_data.A.columns]]
DF_metadata_new.to_csv('../../data/interim/inferred_samples_metadata.csv')

In [ ]:
DF_metadata_inferred = pd.read_csv('../../data/interim/inferred_samples_metadata_curated.csv')

included_samples = ica_data.A.columns

included_projects = set(
    DF_metadata.loc[
        DF_metadata.index.intersection(included_samples),
        "BioProject"
    ]
)

DF_metadata_inferred["Excluded_from_included_project"] = (
    DF_metadata_inferred["BioProject"].isin(included_projects)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count excluded samples by reason and exclusion type
counts = (
    DF_metadata_inferred
    .groupby(["exclusion_reason", "Excluded_from_included_project"])
    .size()
    .unstack(fill_value=0)
)

# Rename columns for nicer legend
counts = counts.rename(columns={
    False: "Entire project excluded",
    True: "Sample excluded (QC)"
})

# Order by total samples
counts = counts.loc[counts.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(9, 6))

counts.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    width=0.8,
    color=["blue", "red"]
)

ax.set_xlabel("Excluded samples")
ax.set_ylabel("")
ax.legend(frameon=False)

plt.tight_layout()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------
# Create one row per project
# ---------------------------------------------
project_summary = (
    DF_metadata_inferred
    .groupby("BioProject")
    .agg(
        exclusion_reason=("exclusion_reason", "first"),
        partial_exclusion=("Excluded_from_included_project", "any")
    )
)

# partial_exclusion == True
#   -> project was included but at least one sample was removed
# partial_exclusion == False
#   -> whole project excluded

counts = (
    project_summary
    .groupby(["exclusion_reason", "partial_exclusion"])
    .size()
    .unstack(fill_value=0)
)

counts = counts.rename(columns={
    False: "Entire project excluded",
    True: "Partial project (sample QC)"
})

# Order by total number of excluded projects
counts = counts.loc[counts.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(9,6))

counts.plot.barh(
    stacked=True,
    ax=ax,
    edgecolor="black",
    width=0.8,
    color=["blue", "red"]
)

totals = counts.sum(axis=1)

for y, total in enumerate(totals):
    ax.text(
        total + 0.15,
        y,
        str(total),
        va="center",
        fontsize=10
    )

ax.set_xlabel("Number of BioProjects")
ax.set_ylabel("")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('images/fig6_project_status.svg', dpi = 600, format = 'svg', bbox_inches='tight')

plt.show()